# Tech Challenge - Fase 2

# Exploração da Camada Gold

## Objetivo

Este notebook apresenta uma análise exploratória dos datasets produzidos pela camada Gold da arquitetura Medalhão implementada no projeto.

A camada Gold consolida os dados tratados nas etapas Bronze e Silver, disponibilizando conjuntos de dados preparados para análises estatísticas, construção de dashboards, aplicações de Machine Learning e atualização incremental via streaming.

Serão explorados os seguintes datasets:

- indicador_municipio.parquet
- comparativo_metas.parquet
- evolucao_temporal.parquet
- alunos_modelagem.parquet

## Estrutura do notebook

O notebook está organizado nas seguintes etapas:

1. Configuração do ambiente
2. Carregamento dos datasets
3. Funções auxiliares
4. Visão geral da camada Gold
5. Análise do Indicador por Município
6. Análise do Comparativo de Metas
7. Análise da Evolução Temporal
8. Análise do Dataset para Machine Learning
9. Integração com Streaming
10. Conclusões

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)
pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
PROJECT_ROOT = Path("..").resolve()

GOLD_PATH = PROJECT_ROOT / "data" / "gold"

print(GOLD_PATH)

/home/erick/Documents/FIAP/fase_2/tech_challenge/projeto/tech_challenge_2_alfabetizacao/data/gold


## Carregamento dos datasets

In [3]:
indicador = pd.read_parquet(
    GOLD_PATH / "indicador_municipio.parquet"
)

comparativo = pd.read_parquet(
    GOLD_PATH / "comparativo_metas.parquet"
)

evolucao = pd.read_parquet(
    GOLD_PATH / "evolucao_temporal.parquet"
)

alunos = pd.read_parquet(
    GOLD_PATH / "alunos_modelagem.parquet"
)

## Funções auxiliares

In [4]:
def resumo_dataset(df: pd.DataFrame, nome: str) -> pd.DataFrame:
    """
    Retorna um resumo geral do dataset.
    """

    return pd.DataFrame(
        {
            "Dataset": [nome],
            "Registros": [len(df)],
            "Colunas": [df.shape[1]],
            "Memória (MB)": [
                round(df.memory_usage(deep=True).sum() / 1024**2, 2)
            ],
        }
    )

In [5]:
def analisar_nulos(df: pd.DataFrame) -> pd.DataFrame:

    resultado = (
        df.isna()
          .sum()
          .rename("Valores Nulos")
          .to_frame()
    )

    resultado["Percentual"] = (
        resultado["Valores Nulos"] / len(df) * 100
    )

    return resultado.sort_values(
        "Valores Nulos",
        ascending=False,
    )

In [6]:
def analisar_tipos(df: pd.DataFrame) -> pd.DataFrame:

    return pd.DataFrame(
        {
            "Tipo": df.dtypes.astype(str),
            "Não Nulos": df.count(),
        }
    )

# Visão Geral da Camada Gold

In [7]:
visao_geral = pd.concat(
    [
        resumo_dataset(indicador, "Indicador Município"),
        resumo_dataset(comparativo, "Comparativo Metas"),
        resumo_dataset(evolucao, "Evolução Temporal"),
        resumo_dataset(alunos, "Alunos Modelagem"),
    ],
    ignore_index=True,
)

visao_geral

,Dataset,Registros,Colunas,Memória (MB)
0,Indicador Município,23995,15,6.77
1,Comparativo Metas,75516,11,26.02
2,Evolução Temporal,24143,10,8.51
3,Alunos Modelagem,3867999,24,2713.95


### Considerações iniciais

A camada Gold é composta por quatro datasets especializados, cada um destinado a um propósito analítico específico:

- **Indicador Município:** consolida os indicadores municipais de alfabetização.
- **Comparativo Metas:** permite comparar metas entre os diferentes níveis geográficos.
- **Evolução Temporal:** disponibiliza a evolução histórica dos indicadores.
- **Alunos Modelagem:** reúne atributos preparados para futuras aplicações de Machine Learning.

Essa separação reduz redundâncias, simplifica as consultas analíticas e mantém cada conjunto de dados focado em um objetivo específico.

# Análise do Dataset: Indicador por Município

O dataset **indicador_municipio.parquet** representa o principal produto analítico da camada Gold.

Cada registro corresponde ao desempenho de uma combinação de:

- município;
- ano;
- rede de ensino.

Esse dataset foi modelado para suportar consultas analíticas, dashboards e atualizações incrementais via streaming.

In [8]:
resumo_dataset(indicador, "Indicador Município")

,Dataset,Registros,Colunas,Memória (MB)
0,Indicador Município,23995,15,6.77


In [9]:
analisar_tipos(indicador)

,Tipo,Não Nulos
ano,Int64,23995
id_municipio,string,23995
municipio,string,23995
rede,string,23995
taxa_alfabetizacao,float64,23995
...,...,...
meta_alfabetizacao_2026,float64,10654
meta_alfabetizacao_2027,float64,10654
meta_alfabetizacao_2028,float64,10654
meta_alfabetizacao_2029,float64,10654


In [10]:
analisar_nulos(indicador)

,Valores Nulos,Percentual
meta_alfabetizacao_2024,13531,56.39
nivel_alfabetizacao,13411,55.89
percentual_participacao,13411,55.89
meta_alfabetizacao_2025,13341,55.60
meta_alfabetizacao_2026,13341,55.60
...,...,...
id_municipio,0,0.00
municipio,0,0.00
rede,0,0.00
taxa_alfabetizacao,0,0.00


### Avaliação da qualidade dos dados

Os valores ausentes observados concentram-se nas colunas relacionadas ao indicador de alfabetização.

Esses registros representam municípios ou combinações de ano/rede para os quais ainda não existem medições disponíveis, não caracterizando necessariamente inconsistências no processo ETL.

## Cobertura geográfica

In [14]:
pd.DataFrame(
    {
        "Municípios únicos": [
            indicador["id_municipio"].nunique()
        ],
        "Anos": [
            indicador["ano"].nunique()
        ],
        "Redes": [
            indicador["rede"].nunique()
        ],
    }
)

,Municípios únicos,Anos,Redes
0,5550,2,4


A distribuição dos registros por Cidade demonstra que o dataset contempla todo o território nacional, permitindo análises comparativas entre estados e municípios.

## Distribuição por rede de ensino

In [16]:
(
    indicador["rede"]
    .value_counts()
    .rename_axis("Rede")
    .to_frame("Quantidade")
)

,Quantidade
Rede,
Municipal,10896
Pública (Estadual e Municipal),10466
Estadual,2235
"Total (Federal, Estadual, Municipal e Privada)",398


Cada município pode possuir indicadores distintos para diferentes redes de ensino, permitindo análises segmentadas do desempenho educacional.

## Estatísticas dos indicadores

In [17]:
indicador[
    [
        "media_portugues",
        "nivel_alfabetizacao",
        "percentual_participacao",
    ]
].describe()

,media_portugues,nivel_alfabetizacao,percentual_participacao
count,23995.00,10584.00,10584.00
mean,751.38,2.67,90.43
std,23.23,1.71,6.27
min,673.30,0.00,70.00
25%,736.04,1.00,86.67
50%,750.65,3.00,91.20
75%,764.30,4.00,95.06
max,868.46,5.00,100.00


In [18]:
indicador[
    [
        "media_portugues",
        "nivel_alfabetizacao",
        "percentual_participacao",
    ]
].agg(
    ["min", "max", "mean", "median"]
)

,media_portugues,nivel_alfabetizacao,percentual_participacao
min,673.30,0.00,70.00
max,868.46,5.00,100.00
mean,751.38,2.67,90.43
median,750.65,3.00,91.20


As estatísticas descritivas permitem avaliar a dispersão dos indicadores educacionais e identificar possíveis valores extremos que mereçam investigação em análises futuras.

## Exemplos de registros

In [19]:
indicador.sample(
    10,
    random_state=42,
)

,ano,id_municipio,municipio,rede,taxa_alfabetizacao,media_portugues,nivel_alfabetizacao,percentual_participacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030
21930,2024,4302501,Bossoroca,Pública (Estadual e Municipal),46.90,735.05,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7417,2023,3532405,Nazaré Paulista,Estadual,61.39,750.02,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7049,2023,3505906,Batatais,Pública (Estadual e Municipal),64.89,755.84,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7524,2023,3541059,Pratânia,Municipal,56.06,745.94,2,86.90,60.03,63.88,67.55,71.02,74.27,77.26,80.00
6930,2023,3305109,São João de Meriti,Pública (Estadual e Municipal),37.60,722.46,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17731,2024,3142601,Monsenhor Paulo,Pública (Estadual e Municipal),78.89,767.24,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22605,2024,4315073,Porto Vera Cruz,Municipal,88.20,774.52,5,100.00,65.93,68.59,71.14,73.56,75.85,77.99,80.00
7760,2023,3556958,Vitória Brasil,Municipal,66.67,757.68,3,100.00,68.83,70.92,72.91,74.82,76.64,78.37,80.00
10243,2023,4317558,Santo Antônio do Palma,Pública (Estadual e Municipal),72.22,764.78,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
536,2023,1507151,São Domingos do Araguaia,Municipal,46.05,740.44,1,79.26,51.56,57.03,62.33,67.36,72.01,76.24,80.00


## Considerações

O dataset **Indicador por Município** constitui o principal produto analítico da camada Gold.

Suas principais características são:

- consolidação dos indicadores educacionais por município;
- estrutura otimizada para consultas analíticas;
- suporte à atualização incremental implementada no módulo de streaming;
- utilização como base para dashboards e monitoramento contínuo dos indicadores de alfabetização.

# Análise do Dataset: Comparativo de Metas

O dataset **comparativo_metas.parquet** consolida as metas de alfabetização nos diferentes níveis geográficos (Brasil, Unidade da Federação e Município).

Sua principal finalidade é permitir análises comparativas entre os níveis administrativos e acompanhar a evolução das metas estabelecidas ao longo do tempo.

In [20]:
resumo_dataset(
    comparativo,
    "Comparativo de Metas",
)

,Dataset,Registros,Colunas,Memória (MB)
0,Comparativo de Metas,75516,11,26.02


In [21]:
analisar_tipos(comparativo)

,Tipo,Não Nulos
nivel_geografico,string,75516
ano_origem,Int64,75516
ano_meta,int64,75516
uf,string,567
id_municipio,string,74928
...,...,...
rede,string,75516
taxa_alfabetizacao,float64,74648
nivel_alfabetizacao,Int64,74088
percentual_participacao,float64,74648


In [22]:
analisar_nulos(comparativo)

,Valores Nulos,Percentual
uf,74949,99.25
nivel_alfabetizacao,1428,1.89
taxa_alfabetizacao,868,1.15
percentual_participacao,868,1.15
id_municipio,588,0.78
...,...,...
meta_alfabetizacao,262,0.35
nivel_geografico,0,0.00
ano_origem,0,0.00
ano_meta,0,0.00


In [23]:
pd.DataFrame(
    {
        "Anos disponíveis": [
            comparativo["ano_meta"].nunique()
        ],
        "Níveis geográficos": [
            comparativo["nivel_geografico"].nunique()
        ],
        "UFs": [
            comparativo["uf"].nunique()
        ],
    }
)

,Anos disponíveis,Níveis geográficos,UFs
0,7,3,27


In [24]:
comparativo[
    "nivel_geografico"
].value_counts().to_frame("Quantidade")

,Quantidade
nivel_geografico,
Município,74928
UF,567
Brasil,21


In [25]:
comparativo[
    "ano_meta"
].value_counts().sort_index().to_frame("Quantidade")

,Quantidade
ano_meta,
2024,10788
2025,10788
2026,10788
2027,10788
2028,10788
2029,10788
2030,10788


### Considerações

O dataset foi estruturado para permitir consultas comparativas entre diferentes níveis geográficos, mantendo uma única estrutura de dados para Brasil, Unidades da Federação e Municípios.

Essa modelagem reduz a necessidade de múltiplas consultas e simplifica a construção de indicadores analíticos.

# Análise do Dataset: Evolução Temporal

O dataset **evolucao_temporal.parquet** reúne os indicadores históricos produzidos pela pipeline.

Seu objetivo é disponibilizar uma visão temporal da evolução da alfabetização, permitindo análises históricas sem necessidade de reprocessamento das camadas Bronze ou Silver.

In [26]:
resumo_dataset(
    evolucao,
    "Evolução Temporal",
)

,Dataset,Registros,Colunas,Memória (MB)
0,Evolução Temporal,24143,10,8.51


In [27]:
analisar_tipos(evolucao)

,Tipo,Não Nulos
nivel_geografico,string,24143
ano,Int64,24143
uf,string,145
id_municipio,string,23995
municipio,string,23995
rede,string,24143
taxa_alfabetizacao,float64,24143
media_portugues,float64,24140
nivel_alfabetizacao,Int64,0
percentual_participacao,float64,3


In [28]:
analisar_nulos(evolucao)

,Valores Nulos,Percentual
nivel_alfabetizacao,24143,100.00
percentual_participacao,24140,99.99
uf,23998,99.40
id_municipio,148,0.61
municipio,148,0.61
media_portugues,3,0.01
nivel_geografico,0,0.00
ano,0,0.00
rede,0,0.00
taxa_alfabetizacao,0,0.00


In [29]:
pd.DataFrame(
    {
        "Anos": [
            evolucao["ano"].nunique()
        ],
        "UFs": [
            evolucao["uf"].nunique()
        ],
        "Municípios": [
            evolucao["id_municipio"].nunique()
        ],
    }
)

,Anos,UFs,Municípios
0,3,25,5550


In [30]:
evolucao[
    "ano"
].value_counts().sort_index().to_frame("Quantidade")

,Quantidade
ano,
2023,11618
2024,12524
2025,1


In [31]:
evolucao.sample(
    10,
    random_state=42,
)

,nivel_geografico,ano,uf,id_municipio,municipio,rede,taxa_alfabetizacao,media_portugues,nivel_alfabetizacao,percentual_participacao
9932,Município,2023,<NA>,4312203,Maximiliano de Almeida,Estadual,83.33,762.39,<NA>,NaN
21459,Município,2024,<NA>,4210308,Major Vieira,Municipal,74.01,758.45,<NA>,NaN
10146,Município,2023,<NA>,4315313,Quatro Irmãos,Municipal,52.94,736.33,<NA>,NaN
16554,Município,2024,<NA>,2932200,Ubaitaba,Pública (Estadual e Municipal),31.13,719.69,<NA>,NaN
3699,Município,2023,<NA>,2802908,Itabaiana,Estadual,31.93,712.69,<NA>,NaN
19969,Município,2024,<NA>,3544301,Roseira,Pública (Estadual e Municipal),67.20,753.23,<NA>,NaN
17948,Município,2024,<NA>,3150109,Piau,Municipal,75.58,753.28,<NA>,NaN
17605,Município,2024,<NA>,3138302,Leandro Ferreira,Estadual,74.29,763.77,<NA>,NaN
18166,Município,2024,<NA>,3158300,Santana da Vargem,Estadual,88.46,776.45,<NA>,NaN
2851,Município,2023,<NA>,2510006,Nazarezinho,Municipal,36.48,732.94,<NA>,NaN


In [40]:
evolucao.loc[evolucao['nivel_geografico'] == 'Brasil']

,nivel_geografico,ano,uf,id_municipio,municipio,rede,taxa_alfabetizacao,media_portugues,nivel_alfabetizacao,percentual_participacao
0,Brasil,2023,<NA>,<NA>,<NA>,Pública,55.90,NaN,<NA>,86.00
1,Brasil,2024,<NA>,<NA>,<NA>,Pública,59.20,NaN,<NA>,87.37
2,Brasil,2025,<NA>,<NA>,<NA>,Pública,66.00,NaN,<NA>,88.00


### Considerações

O dataset foi modelado para disponibilizar séries históricas dos indicadores educacionais.

Sua utilização evita reprocessamentos das camadas inferiores da arquitetura Medalhão e fornece uma base consolidada para análises evolutivas e acompanhamento longitudinal dos indicadores.

# Análise do Dataset: Alunos para Modelagem

O dataset **alunos_modelagem.parquet** foi desenvolvido para disponibilizar uma visão consolidada dos alunos juntamente com os indicadores educacionais produzidos pela camada Gold.

Seu principal objetivo é servir como base para futuras aplicações de Ciência de Dados e Machine Learning, eliminando a necessidade de novas integrações entre diferentes conjuntos de dados.

In [41]:
resumo_dataset(
    alunos,
    "Alunos Modelagem",
)

,Dataset,Registros,Colunas,Memória (MB)
0,Alunos Modelagem,3867999,24,2713.95


In [42]:
analisar_tipos(alunos)

,Tipo,Não Nulos
ano,Int64,3867999
id_municipio,string,3867999
id_municipio_nome,string,3867999
id_escola,string,3867999
id_aluno,string,3867999
...,...,...
meta_alfabetizacao_2026,float64,3348950
meta_alfabetizacao_2027,float64,3348950
meta_alfabetizacao_2028,float64,3348950
meta_alfabetizacao_2029,float64,3348950


In [43]:
analisar_nulos(alunos)

,Valores Nulos,Percentual
meta_alfabetizacao_2024,559165,14.46
percentual_participacao,534671,13.82
nivel_alfabetizacao,534671,13.82
meta_alfabetizacao_2030,519049,13.42
meta_alfabetizacao_2029,519049,13.42
...,...,...
caderno,0,0.00
id_aluno,0,0.00
id_escola,0,0.00
id_municipio_nome,0,0.00


In [48]:
pd.DataFrame(
    {
        "Alunos": [
            len(alunos)
        ],
        "Municípios": [
            alunos["id_municipio"].nunique()
        ],
        "Rede": [
            alunos["rede"].nunique()
        ],
        "Anos": [
            alunos["ano"].nunique()
        ],
    }
)

,Alunos,Municípios,Rede,Anos
0,3867999,5548,3,2


In [49]:
alunos.sample(
    10,
    random_state=42,
)

,ano,id_municipio,id_municipio_nome,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno,taxa_alfabetizacao,media_portugues,nivel_alfabetizacao,percentual_participacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030
230884,2023,2111300,São Luís,60004286,21007616,17,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,47.49,736.94,1,82.48,52.79,58.04,63.10,67.90,72.34,76.38,80.00
84220,2023,1304203,Tefé,60001120,13061387,9,2° ano do Ensino Fundamental,Municipal,Presente,Prova preenchida,Não,736.25,0.84,41.26,728.21,<NA>,NaN,NaN,44.50,52.51,60.40,67.78,74.36,80.00
174143,2023,1506807,Santarém,60001490,15088255,9,2° ano do Ensino Fundamental,Municipal,Presente,Prova preenchida,Não,686.95,1.03,54.92,738.90,2,81.79,59.08,63.11,66.97,70.62,74.01,77.14,80.00
2697251,2024,3205200,Vila Velha,60022698,32014652,4,2° ano do Ensino Fundamental,Municipal,Presente,Prova preenchida,Sim,811.38,1.17,67.75,756.35,3,88.97,70.00,71.85,73.63,75.34,76.97,78.52,80.00
363975,2023,2307700,Maranguape,60008238,23030334,18,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,79.76,786.21,4,95.16,79.79,79.83,79.86,79.90,79.93,79.97,80.00
1080445,2023,3301900,Itaboraí,60025598,33035771,11,2° ano do Ensino Fundamental,Municipal,Presente,Prova preenchida,Não,739.70,1.32,51.30,735.97,2,85.01,56.04,60.66,65.11,69.31,73.21,76.78,80.00
1228382,2023,4122305,Rio Negro,60034241,41023711,14,2° ano do Ensino Fundamental,Municipal,Presente,Prova preenchida,Sim,795.00,1.20,85.17,769.41,5,92.43,80.00,80.00,80.00,80.00,80.00,80.00,80.00
1355952,2023,4202800,Braço do Norte,60035000,42012749,11,2° ano do Ensino Fundamental,Municipal,Presente,Prova preenchida,Sim,743.11,2.29,60.64,746.05,3,78.77,63.84,66.93,69.87,72.66,75.28,77.73,80.00
3141008,2024,3518800,Guarulhos,60026754,35259714,19,2° ano do Ensino Fundamental,Municipal,Presente,Prova preenchida,Sim,745.70,1.07,44.19,733.02,1,82.99,45.75,52.23,58.63,64.75,70.42,75.53,80.00
3517306,2024,4209102,Joinville,60034793,42053276,16,2° ano do Ensino Fundamental,Municipal,Presente,Prova preenchida,Sim,768.24,1.42,68.79,756.24,3,71.61,68.23,70.43,72.54,74.56,76.48,78.29,80.00


### Considerações

O dataset foi enriquecido com informações provenientes do principal indicador analítico da camada Gold, disponibilizando uma estrutura preparada para futuras aplicações de Machine Learning.

Essa abordagem reduz a necessidade de novas etapas de preparação dos dados e simplifica o desenvolvimento de modelos analíticos.

# Integração com Streaming

Além da geração dos datasets analíticos por meio da pipeline ETL, o projeto implementa uma camada de streaming utilizando Apache Kafka.

A solução permite simular atualizações dos indicadores educacionais em tempo quase real, demonstrando a integração entre processamento batch e processamento orientado a eventos.

In [50]:
from pathlib import Path

streaming_files = [
    "run_streaming.py",
    "run_consumer.py",
    "producer.py",
    "consumer.py",
    "update_gold.py",
]

pd.DataFrame(
    {
        "Componente": streaming_files,
        "Disponível": [
            (PROJECT_ROOT / "streaming" / f).exists()
            for f in streaming_files
        ],
    }
)

,Componente,Disponível
0,run_streaming.py,True
1,run_consumer.py,True
2,producer.py,True
3,consumer.py,True
4,update_gold.py,True


## Fluxo implementado

O módulo de streaming segue o fluxo abaixo:

```
Simulador
      │
      ▼
Producer Kafka
      │
      ▼
Apache Kafka
      │
      ▼
Consumer
      │
      ▼
Atualização do indicador_municipio.parquet
```

Com essa arquitetura, o dataset **indicador_municipio.parquet** deixa de depender exclusivamente da execução completa da pipeline ETL, podendo receber atualizações incrementais provenientes de eventos publicados no Kafka.

# Conclusões

A camada Gold representa a etapa final da arquitetura Medalhão implementada no projeto.

Os datasets produzidos foram modelados para atender diferentes necessidades analíticas:

- **indicador_municipio.parquet**: consolidação dos indicadores educacionais.
- **comparativo_metas.parquet**: comparação entre metas e níveis geográficos.
- **evolucao_temporal.parquet**: acompanhamento histórico dos indicadores.
- **alunos_modelagem.parquet**: preparação dos dados para aplicações de Machine Learning.

Além da pipeline batch, foi implementado um módulo de streaming baseado em Apache Kafka capaz de atualizar incrementalmente o principal dataset analítico da camada Gold, demonstrando a integração entre processamento em lote e processamento orientado a eventos.

Com essa arquitetura, o projeto entrega uma solução modular, escalável e alinhada às boas práticas de Engenharia de Dados.